### Databricks Assets Management through **Policy** in Unity Catalog 

### Full Policy Workflow

In [ ]:
* STEP 1: Structure of a tag and its creation is discussed.
* STEP 2: Assignment of tags to securable data assets such as catalog, schema, tables, volumes etc.
* STEP 3: Policy: Automation on Databricks Access Control.
* STEP 4: Row Filter and Column Mask

First, better ask the question  
What is a **tag** in Unity Catalog ? 

A tag is a key–value attribute that you can attach to securable objects (such as catalogs, schemas, tables, views, functions, etc.)

What is the purpose of using tags ?   
To help  
* organize data assets. 
* classify data assets. 
* centrally govern data assets.

#### Structure of a TAG  

**Key**: The name of the tag. As for an example, 'environment' is a key.  
**Value**: The values are 'dev', 'qa' or 'prod'.  
Thus we have key:value pairs as follows:  

'environment': 'dev'  
'environment': 'qa'    
'environment': 'prod'  

### Tag or Governed Tag ?

In [ ]:
In Databricks the tags are called **Goverened Tags**.    

**In this document we will discuss governed tags only.**

## STEP 1:

**Creation of Tags**:     
You can create tags from catalog browser or from writing SQL syntax.    
In this document, we will create tags using SQL syntax.  

In [ ]:
%sql
CREATE GOVERNED TAG key_tag_name;

The above SQL statement creates a tag "key_tag_name". It is **governed** by Unity Catalog.  
The above tag name is unique in a Databricks account. If you try to create a tag with the same name, Databricks will throw an "ALREADY_EXISTS" error.    
Those users have CREATE permission, can create tags. Databricks admin has a CREATE permission.   
In an organization there are several tags needed for Assets Management. That is why tag creation is handled by the 'admin' user.   
Otherwise, tags swamps will be created.      

Tag creation takes place at **account level**. So, the scope of any tag is anywhere in Unity Catalog based governed Assets. Thus, a tag may be assigned to a catalog, a schema, a table and so on.

In [ ]:
from IPython.display import display, HTML

text_to_box = "This is a dynamically generated box from Python code!"

box_html = f"""
<div style="border: 3px double #0076bb; padding: 15px; background-color: #f4f9fc; text-align: center;">
    <p style="font-family: Arial, sans-serif; color: #333;">{text_to_box}</p>
</div>
"""

display(HTML(box_html))

In [5]:
from IPython.display import display, HTML

def draw_box(text, bg_color="#f4f9fc", border_color="#0076bb", border_style="solid", border_width="2px", padding="15px", radius="5px"):
    """
    Displays a customizable HTML box around text in Jupyter Lab.
    """
    box_html = f"""
    <div style="
        display:inline-block;
        background-color: {bg_color}; 
        border: {border_width} {border_style} {border_color}; 
        padding: {padding}; 
        border-radius: {radius};
        margin: 10px 0;
        font-family: Arial, sans-serif;
    ">
        {text}
    </div>
    """
    display(HTML(box_html))

In [14]:
draw_box("<span style='color: red;'>Important! </span>&nbsp;&nbsp;  A tag is an ACCOUNT LEVEL RESOURCE")

A tag is associated with a key:value pair. Now we want to  "Create a tag with its values".

In [ ]:
CREATE GOVERNED TAG pii VALUES('email');

'pii' means personal indentifiable information. Here, 'email' is a value that 'pii' tag can take.  

#### A Tag with Multiple Values

In [ ]:
CREATE GOVERNED TAG sensivity VALUES('low','medium','high');

A tag 'sensivity' is created with three values 'low', 'medium' and 'high' that it can take.

#### A Tag with empty value ! 

In [ ]:
CREATE GOVERNED TAG key_tag VALUES();

We shortly see the application of an empty tag. 

Now we know how to create tags. Next we want to learn how to apply tags to data objects, viz., catalog, schema, tables, volumes and so on.

## STEP 2:

**Assignment of tags to Data Objects**

Apply a tag on a catalog: 

Let us create a catalog `prod`

In [ ]:
%sql 
CREATE CATALOG IF NOT EXISTS prod

In [ ]:
Assignment of a tag to the catalog 'prod':

In [ ]:
%sql 
SET TAG CATALOG prod `cii` = `hr`

Here, the tag_name (key) is 'cii' and 'hr' is its value. The tag is applied on CATALOG prod. 

Now click the **catalog** on the left navigation bar of Databricks Work Space. Then click the catalog 'prod' and you will see the catalog is tagged on the right.

**But** we want to see 'prod' catalog is tagged from our **program**.

When the catalog 'prod' is created, by default two schemas or databases are created. They are 'default' and **'information schema'**. Now we issue a query on the database 'information_schema'. The database 'information schema' has informations about tagging data-objects in Databricks.

In [ ]:
%sql
select catalog_name, tag_name, tag_value
from prod.information_schema.catalog_tags
where catalog_name='prod';

In a similiar way we can create schemas or databases and tables and assign tag to these data objects.

In [ ]:
%sql 
-- for schema or database
CREATE SCHEMA IF NOT EXISTS retails;
SET TAG on SCHEMA retails `sii` = 'cst' 

In [ ]:
%sql 
-- display the tags on schema/database
SELECT schema_name, tag_name, tag_value 
FROM prod.information_schema.schema_tags 
where schema_name='prod.retails';

In [ ]:
%sql 
-- for a table 
CREATE TABLE IF NOT EXISTS prod.retails.customers (
    id int NOT NULL, 
    first_name STRING,
    last_name STRING,
    email     STRING, 
    address   STRING
) 
USING DELTA;

In [ ]:
%sql 
-- display the tags on the table 
SELECT table_name, tag_name, tag_value 
FROM prod.information_schema.table_tags 
WHERE table_name='customers';

## STEP 3

### Policy: Automation on Databricks Access Control

Policy is an object in Databricks. Data policies define a set of rules that **automate** governed tasks for data access, security and usuage. Policies does so through the use of **tags**, **classifications** and other attributes. Policies provides us help on  
* centralized Governance of sensitive data. Only authorized users are allowed to access the sensitive data.  
* Protection of Data through encryption and masking.  
* Implementation of audit logging and traceability to meet compliance requirements.

Benifits of using Policies and Tags: 
* Data owners **intent** at the object level (self-document).
* Central automation reads tags and acts — no per-table runbooks
* Policy changes are made by **updating** a tag, not rewriting jobs
* Full audit trail via UC system tables

### Create Policy

**CREATE [OR REPLACE] POLICY** policy_name    
**ON { CATALOG** catalog_name | **SCHEMA** schema_name | **TABLE** table_name }   
[**COMMENT** description]   
{ row_filter_body | column_mask_body }   

To create a policy we need few clauses.  
They are :   
* **ON**  clause : Policy applies on catalog|schema|table.
* **FOR** clause : Policy applies on tables { row_filter_body | column_mask_body }
* **TO**  clause : To whom we apply policy. Policy applies on principal.
               principal means a user, group, or service principal name.
* **Except** clause: It is part of a TO clause. After **Except** we have a user|group|principal name.   
                     **Except** means the policy does not apply on user|group|principal name

In [ ]:
<span style = 'color: red;'>Important!</span>&nbsp;&nbsp; A policy is applied ultimately on rows and columns.

In [ ]:
In this context there are two objects:  
* Row Filter Body    
* Column Mask Body  

## Row Filter Body

Syntax of Row Filter Body : [Databricks Document](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-policy)

row_filter_body  
  ROW FILTER function_name  
  TO principal [, ...]   
  [ EXCEPT principal [, ...] ]   
  FOR TABLES    
  [ WHEN condition ]   
  [ MATCH COLUMNS condition [ [ AS ] alias ] [, ...] ]   
  [ USING COLUMNS ( function_arg [, ...] ) ]   

Explanation of the above syntax of row_filter body :  

**ROW FILTER function_name** : "function_name" is a function created by "create function" syntax. create function creates an **UDF** (SQL).  

**WHEN condition :** condition is in the form of has_tag_value('value'). for whom?  

**MATCH COLUMNS condition :** condition is in the form of has_tag('') and then put an alias. Thus all the columns which has matched tags are given a single name or alias.   

This alias is passed to **USING COLUMNS (alias)**


Now we will see several examples to show how rows are **filtered out** using the above syntax.

### **Example 1:**      

### Single table, Single Principal

Let us dive into **create policy** statement. 

In [ ]:
**Assumptions:**
   - **gold** is a catalog.
   - **customers** is a database under the catalog 'gold'.   
   - **orders** is a table in the database 'gold.customers'.     
   - sp_us_analytics is a principal i.e an user|a group|a service principal. 
   - 'geo_region' is a single column of the table customers.orders.   
   - non_eu_region is the name of the **UDF**. This UDF operates on the column 'geo_region'  

In [ ]:
%sql
CREATE POLICY hide_eu_rows
ON TABLE gold.customers.orders
ROW FILTER non_eu_region
TO sp_us_analytics
FOR TABLES
USING COLUMNS (geo_region);

Below, we define the **UDF** non_eu_region() as follows. The **UDF** operates on the column 'geo_region'. It returns True when the column value of geo_region is not 'eu'.

In [ ]:
%sql 
-- udf non_eu_region(geo_region)
CREATE or REPLACE function non_eu_region(geo_region)  
RETURNS BOOLEAN
RETURN geo_region <> 'eu'

#### How the **UDF** works:

When the service principal, sp_us_analytics,  runs a query like SELECT * FROM gold.customers.orders, Databricks automatically and seamlessly injects the row filter into the query's WHERE clause.  

- **The Filter Logic:** The function checks if geo_region <> 'eu'.

- **The Result:** If a row has 'eu' as its region, the function returns FALSE, and Databricks silently filters that row out. If it has 'us', 'apac', etc., it returns TRUE, and the row is displayed.

Thus, the query changes underhood **from**

In [ ]:
SELECT * FROM gold.customers.orders

**To:**

In [ ]:
SELECT * FROM (
  SELECT * FROM gold.customers.orders 
  WHERE non_eu_region(region) -- Enforced at the engine level
);

Thus, all the rows "where geo_region='eu'" are **FILTERED OUT**.The service principal 'sp_us_analytics' is not provided the access of these rows.   

**Finally:**

In [ ]:
%sql 
-- udf non_eu_region(geo_region)
CREATE or REPLACE function non_eu_region(geo_region)  
RETURNS BOOLEAN
RETURN geo_region <> 'eu'

CREATE POLICY hide_eu_rows
ON TABLE gold.customers.orders
ROW FILTER non_eu_region
TO sp_us_analytics
FOR TABLES
USING COLUMNS (geo_region);

### **Example 2:**

### Single table, **Multiple**  Principals with an **exception**:

In [ ]:
%sql
CREATE POLICY hide_eu_rows
ON TABLE gold.customers.orders
ROW FILTER non_eu_region
TO sp_us_analytics, sp_eu_readonly
EXCEPT sp_data_governance_admin
FOR TABLES
USING COLUMNS (geo_region);

* We have added two service principals comma separated. 
* **EXCEPT** carves out a principal that should bypass the filter — useful for an **admin/governance SP** that needs **unfiltered** access.

### **Example 3:**  

### **Multiple tables** — schema-level policy using **tag** matching. 

In Example 2, we have used only one table (customers.orders) and one column (geo_region). So, the power of **FOR TABLES** clause is not used
in a proper manner.   
Now we introduce two clauses after *FOR TABLES* and they are known as **CONDITION**:  
* **WHEN**
* **MATCH COLUMNS**  

Both the clauses use tags attached to data objects. Thus we find below modified **FOR TABLES**:  

FOR TABLES  
WHEN has_tag_value('sensitivity', 'high')   
MATCH COLUMNS has_tag('geo_region') AS region   
USING COLUMNS(region)   

EXPLANATIONS:      
* `WHEN has_tag_value('sensitivity', 'high')` — only tables tagged **sensitivity=high** get this policy at all. Untagged tables are untouched.  
* MATCH COLUMNS find those columns of the above tagged tables, which has tag 'geo_region'.       
* Pass these columns to the clause **USING COLUMNS()**.     

**Finally:**

In [ ]:
CREATE POLICY hide_eu_rows
ON SCHEMA gold.customers
COMMENT 'Exclude EU rows from sensitive tables for US analyst SPs'
ROW FILTER non_eu_region
TO sp_us_analytics, sp_eu_readonly
EXCEPT sp_data_governance_admin
FOR TABLES
WHEN has_tag_value('sensitivity', 'high')
MATCH COLUMNS has_tag('geo_region') AS region
USING COLUMNS (region);

### **Policy Inheritance :**

We have created the policy on SCHEMA **gold.customers** and this policy can be applied on all the tables under the schema 'customers'. Thus the policy is <span style='color:red;'>inherited</span> from the schema to all the tables (has the tag 'sensivity' with the 'high' value). 

Unity Catalog has a three level namespace. Policy created **ON** catalog can be inherited to all the **databases** under the catalog and also to all the **tables** under each schema or database.

<span style='color:red;'>Warning:</span> Policy can not be inherited to columns or rows of a table from catalog|schema|table.

### **Example 4:**     
### Catalog-wide policy, many tables, many principals


Scale it up one more level — attach at the catalog so it spans every schema and table underneath:

In [ ]:
%sql   
CREATE POLICY hide_eu_rows
ON CATALOG gold
COMMENT 'ABAC row filter: exclude EU rows across all tagged tables'
ROW FILTER non_eu_region
TO sp_us_analytics, sp_eu_readonly, sp_reporting_svc
EXCEPT sp_data_governance_admin, sp_platform_owner
FOR TABLES
WHEN has_tag_value('sensitivity', 'high')
MATCH COLUMNS has_tag('geo_region') AS region
USING COLUMNS (region);

Since this attaches at gold catalog level, every Silver/Gold table under it that's tagged sensitivity=high and has a column tagged geo_region picks up the filter automatically — you don't touch individual tables at all. 

This pairs naturally with the policy-on-tags automation you were building earlier: tag governance drives access enforcement, rather than per-table ALTER TABLE command.

<span style='color:red;'>Important:</span>   

`has_tag()` / `has_tag_value()` are the only functions Databricks currently allows inside WHEN/MATCH COLUMNS conditions — you can't write arbitrary boolean logic there beyond tag checks.



**Neither the policy definition nor the tags are "transmitted" to the compute cluster as-is. What actually gets sent is the resolved output of evaluating them.**    

### Here is the **flow:**

1. **Policy evaluation happens inside Unity Catalog (the control plane), not on the cluster.**   
When a user runs a query, Unity Catalog itself does the work: it looks at the querying user's identity and group memberships, then evaluates table and column conditions against the tags on the queried object, including inherited tags. So your hide_eu_rows policy's tag condition (has_tag_value('sensitivity', 'high') + has_tag('geo_region')) is checked entirely within UC's metadata layer.
 
2. **Tags themselves never leave Unity Catalog — they're just used to decide which filter applies.**     
The governed tags on your table/columns (like geo_region, sensitivity) are metadata that live in UC. They're read during evaluation, but they aren't shipped to the cluster. Only the outcome — the specific row filter to apply — gets passed downstream.

3. **What actually goes to the Databricks Runtime is the "effective" row filter, as part of table metadata.**  
Once UC determines your policy applies to a given query, it determines the effective row filter or column mask and sends it to the Databricks Runtime as part of the table metadata.   

4. **The Runtime enforces it by rewriting the query plan.**   
On the compute side, the Databricks Runtime query planner translates the effective row filter or column mask into secure view on top of the table scans that enforce filtering and masking during query execution — the same enforcement mechanism used for table-level row filters and column masks.


### **Important**

* **Account admin** or workspace **admin permissions** (to create governed tags).

* **MANAGE permission** on the target catalog or schema.

* **EXECUTE permission** on the UDFs.


Next we discuss **Column Mask Body**. 

# Section Column Mask Body

column_mask_body  
  COLUMN MASK function_name   
  TO principal [, ...]   
  [ EXCEPT principal [, ...] ]   
  FOR TABLES  
  [ WHEN condition ]  
  [ MATCH COLUMNS condition [ [AS] alias ] [, ...] ]   
  ON COLUMN alias   
  [ USING COLUMNS ( function_arg [, ...] ) ]   

Like **Row Filter**, **Column Mask** has several use cases.

## **Use Case 1:**

### **Problem:**    
Suppose there is a **customers** table  and the table has a column "ssn". We want to mask the column so that no user except governance admins **(sp_data_governance_admin)** can view the text in the column "ssn".   
We will write the code in such a way that it can be executed on a **free-edition Databricks Account**. Databricks free-edition account does not allow to create any principal (an user|user group|service principal).   


In [ ]:
%sql
--- Step 1: Initialization
CREATE CATALOG IF NOT EXISTS demo_catalog;
CREATE SCHEMA IF NOT EXISTS demo_catalog.demo_schema;
USE CATALOG demo_catalog;
USE SCHEMA demo_schema;

In [ ]:
%sql 
--- Step 2: Create a table **customers** with a column "email". This column "email" is a sensitive column. 
CREATE OR REPLACE TABLE demo_catalog.demo_schema.customers (
  customer_id INT,
  name STRING,
  email STRING,
  ssn STRING
);

In [ ]:
%sql 
--- Step 3:  
INSERT INTO customers VALUES
  (1, 'Richard Feynman', 'frichard@aol.com', '123-45-6789'),
  (2, 'Robert Oppenheimer', 'orobert@aol.com','987-65-4321'),
  (3, 'Albert Einstein',  'ealbert@yahoo.com', '585-95-1056');


In [ ]:
-- Step 4: Create the masking function
-- This checks the querying user's email. Replace with YOUR actual login email
-- to prove the "unmasked" branch, then swap in a fake one to see it mask.
CREATE OR REPLACE FUNCTION mask_ssn(ssn STRING)
RETURNS STRING
RETURN CASE
  WHEN email LIKE '%@aol.com' THEN ssn
  ELSE '***-**-****'
END;

In [ ]:
-- 4. Apply the mask to the column
ALTER TABLE demo_catalog.demo_schema.customers ALTER COLUMN ssn SET MASK mask_ssn;

In [ ]:
-- 5. Test the code and view the column 'ssn'. 
SELECT * FROM customers;

Please download the ipython notebook for the above solution. [click me for the notebook](Column_Masking_1.ipynb) 

### References:

1.[Databricks Unity Catalog Tags: Governance Best Practices & Patterns](https://community.databricks.com/t5/technical-blog/databricks-unity-catalog-tags-governance-best-practices-amp/ba-p/146575)

2.[Create and manage row filter and column mask policies](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies)

3.[CREATE GOVERNED TAG](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-governed-tag)

4.[A Primer on Leveraging Tags in Databricks for Data + AI Asset Governance](https://community.databricks.com/t5/technical-blog/a-primer-on-leveraging-tags-in-databricks-for-data-ai-asset/ba-p/119908)